In [1]:
import pandas as pd

In [9]:
green_vars = pd.read_csv('SP500_Characteristics.csv')

In [10]:
green_vars['date'] = pd.to_datetime(green_vars['date'])

# 1. Shift the date back 1 month so it accurately reflects when the characteristics were known
green_vars['date'] = green_vars['date'] - pd.DateOffset(months=1)
green_vars['date'] = green_vars['date'] + pd.tseries.offsets.MonthEnd(0)

# 2. Rename the return column (Since the date is now 12/31/21, the Jan 2022 return is exactly ret_fwd_1)
green_vars = green_vars.rename(columns={'ret': 'ret_fwd_1', 'date': 'datadate'})

In [12]:
features = ['mve', 'mom12m', 'bm'] 

# 3. Cross-Sectional Normalization (Mean 0, Std 1) & Imputation
def normalize_and_impute(group):
    # Standardize ignoring NaNs
    means = group[features].mean()
    stds = group[features].std(ddof=0)
    stds = stds.replace(0, 1) # Prevent division by zero
    
    standardized = (group[features] - means) / stds
    
    # Impute any missing characteristics with 0 (the cross-sectional mean)
    return standardized.fillna(0)

# Apply the normalization month-by-month
green_vars[features] = green_vars.groupby('datadate', group_keys=False).apply(normalize_and_impute)

# 4. Final Cleanup: Drop rows where the target variable (forward return) is missing
# (e.g., the very last month in the dataset, since we don't have its future return yet)
green_vars = green_vars.dropna(subset=['ret_fwd_1'])

/var/folders/w3/6myr5tjs6ls3zn1g3ywbc29r0000gn/T/ipykernel_41074/788684481.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  green_vars[features] = green_vars.groupby('datadate', group_keys=False).apply(normalize_and_impute)


In [16]:
green_vars[green_vars['datadate']=='2025-11-30']

,PERMNO,ticker,comnam,datadate,ret_fwd_1,mve_m,mve,mom12m,bm
23650,10104,ORCL,ORACLE CORP,2025-11-30,-0.034860,5.600018e+08,2.247101,0.369090,-1.019797
23651,10107,MSFT,MICROSOFT CORP,2025-11-30,-0.017052,3.594447e+09,3.242669,0.207029,-0.829574
23652,10138,TROW,T ROWE PRICE GROUP INC,2025-11-30,0.012256,2.234310e+07,-0.646102,-0.578353,0.334447
23653,10145,HON,HONEYWELL INTERNATIONAL INC,2025-11-30,0.015089,1.238601e+08,0.892039,-0.643660,-0.666905
23654,10516,ADM,ARCHER DANIELS MIDLAND CO,2025-11-30,-0.053507,2.762797e+07,-0.455423,0.454320,2.128098
...,...,...,...,...,...,...,...,...,...
24148,93096,DG,DOLLAR GENERAL CORP NEW,2025-11-30,0.212622,2.922520e+07,-0.404946,1.233667,0.560821
24149,93132,FTNT,FORTINET INC,2025-11-30,-0.021201,5.905309e+07,0.226792,-0.856774,-1.042215
24150,93246,GNRC,GENERAC HOLDINGS INC,2025-11-30,-0.100640,8.002737e+06,-1.568226,-0.456886,-0.161653
24151,93429,CBOE,C B O E GLOBAL MARKETS INC,2025-11-30,-0.027772,2.626590e+07,-0.500829,0.745907,-0.375912


In [17]:
green_vars.to_csv('green cleaned new.csv', index=False)